# Hack-A-Roo 2026, AI for Quantum track
# Notebook 5: Repetition-code decoder

**Problem.** A quantum error-correcting code does not fix errors by itself. It produces a syndrome, a few measured bits that say where the stabilizers disagree. A decoder reads the syndrome and guesses the error. The decoder is classical software, it has to be fast, and for big codes it cannot be a lookup table. Can a neural network learn to decode?

**What this notebook already does.** It builds the distance-d repetition code as a real qiskit circuit (data qubits, ancilla qubits, CX syndrome extraction, measurement), applies bit-flip noise to the data qubits, simulates it with Aer, and records the syndrome and the true error for every shot. It then builds three decoders: a minimum-weight lookup table, an exact O(d) chain decoder that does the same job without the table, and a small neural network trained on the shots. Each one is scored across a sweep of physical error rates.

**Metrics.** The primary number is the recovery failure rate: the fraction of shots where applying the decoder's correction does not return the data block to the all-zeros codeword. Under perfect syndrome measurement a sensible correction leaves either no residual (success) or an all-ones residual (a logical X error); any other residual means the decoder left the codespace, and that is counted as a failure too. Alongside it, the syndrome consistency rate: the fraction of corrections whose own syndrome matches the measured one. A decoder that returns garbage fails both.

**Why not just majority-vote the corrected block?** Because for this code under X-only noise, the majority vote of the corrected block equals the majority vote of the raw block for every error pattern, whether you correct with the minimum-weight decoder or do nothing at all. That number measures the code, not the decoder. It is kept at the bottom of the results as a labeled classical readout baseline, with a no-correction row to prove the point.

**What to expect.** With independent bit-flip noise and perfect syndrome measurement, minimum-weight decoding is optimal for the repetition code, so the neural decoder can at best tie it. That is the starting line, not a flaw. Several shred-outs below break one of those two assumptions; the distance sweep does not, and says so.

**Deliverable for the poster.** The logical-error suppression curve (recovery failure versus physical error rate, log-log, with the unencoded line and the exact analytic curve) and one number per decoder at the training rate. Then whichever shred-out you pick at the bottom.

Run every cell top to bottom once. The base run takes under a minute. Then go to the Shred-outs section.

In [ ]:
# CONFIG CELL. Every constant, seed, and path lives here. Nothing is hardcoded downstream.

RANDOM_SEED = 42

# Code
CODE_DISTANCE = 5                       # odd. Data qubits = CODE_DISTANCE, ancilla qubits = CODE_DISTANCE - 1
SYNDROME_ROUNDS = 1                     # rounds of syndrome extraction. Base run uses 1 with perfect measurement.

# Noise. Independent X (bit-flip) error on each data qubit before syndrome extraction.
TRAINING_PHYSICAL_ERROR_RATE = 0.10
EVALUATION_PHYSICAL_ERROR_RATES = [0.01, 0.02, 0.05, 0.08, 0.10, 0.15, 0.20, 0.25, 0.30, 0.40]

# Shots
TRAINING_SHOTS = 40000
EVALUATION_SHOTS = 40000               # per physical error rate. At p = 0.01 the expected failure count is below 1; see the figure cell.

# Neural decoder
HIDDEN_WIDTH = 64
TRAINING_EPOCHS = 30
BATCH_SIZE = 256
LEARNING_RATE = 1e-3

# Output
OUTPUT_PATH = "repetition_decoder_dataset.pkl"
FIGURE_PATH = "repetition_decoder_figure.png"

In [ ]:
# Install. Skip this cell if your environment already has these. Tested with qiskit 2.6, qiskit-aer 0.18, torch 2.14 (CPU build is enough).
!pip install -q "qiskit[visualization]" "qiskit-aer>=0.17" torch matplotlib

In [ ]:
import time
import pickle
import itertools
import numpy as np
import torch
import qiskit
import qiskit_aer
import matplotlib.pyplot as plt
from scipy.stats import binom
from qiskit import QuantumCircuit, ClassicalRegister, QuantumRegister
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, pauli_error

random_generator = np.random.default_rng(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
assert CODE_DISTANCE % 2 == 1, "the repetition code needs an odd distance so majority vote is well defined"
DATA_QUBIT_COUNT = CODE_DISTANCE
ANCILLA_QUBIT_COUNT = CODE_DISTANCE - 1
SYNDROME_BIT_COUNT = ANCILLA_QUBIT_COUNT * SYNDROME_ROUNDS
package_versions = {"numpy": np.__version__, "qiskit": qiskit.__version__, "qiskit-aer": qiskit_aer.__version__, "torch": torch.__version__}
print("imports done. distance", CODE_DISTANCE, "code on", DATA_QUBIT_COUNT + ANCILLA_QUBIT_COUNT, "qubits,",
      SYNDROME_BIT_COUNT, "syndrome bits", package_versions)

In [ ]:
# The repetition code circuit, built gate by gate.
#
# Layout: data qubits d_0 ... d_{n-1} in a line, ancilla a_i sits between d_i and d_{i+1}.
# Stabilizer i is Z_i Z_{i+1}. It is measured by cx(d_i, a_i), cx(d_{i+1}, a_i), measure a_i.
# The ancilla reads 1 exactly when d_i and d_{i+1} disagree.
#
# The logical state is |0_L> = |00...0>, so every data qubit that reads 1 at the end IS an error.
# That is how every shot gets a free ground-truth label: measuring the data qubits reveals the error pattern.
# This only works because the noise is X-only and the readout is in the Z basis. The repetition code does not
# protect against Z errors at all, and a Z-basis readout cannot see them. Shred-out 7 is where that matters.
#
# Noise is applied with an identity gate on each data qubit that the noise model turns into a bit-flip channel.
# Nothing else in the circuit is noisy in the base run. Noisy CX gates and noisy measurement are shred-outs.

def build_repetition_code_circuit():
    """Logical zero, bit-flip noise on data, SYNDROME_ROUNDS rounds of Z Z stabilizer measurement, final data readout"""
    data_qubits = QuantumRegister(DATA_QUBIT_COUNT, "data")
    ancilla_qubits = QuantumRegister(ANCILLA_QUBIT_COUNT, "ancilla")
    syndrome_bits = ClassicalRegister(SYNDROME_BIT_COUNT, "syndrome")
    data_bits = ClassicalRegister(DATA_QUBIT_COUNT, "final")
    circuit = QuantumCircuit(data_qubits, ancilla_qubits, syndrome_bits, data_bits)

    # the noise model attaches a bit-flip channel to this identity gate
    for data_index in range(DATA_QUBIT_COUNT):
        circuit.id(data_qubits[data_index])

    for round_index in range(SYNDROME_ROUNDS):
        for stabilizer_index in range(ANCILLA_QUBIT_COUNT):
            circuit.cx(data_qubits[stabilizer_index], ancilla_qubits[stabilizer_index])
            circuit.cx(data_qubits[stabilizer_index + 1], ancilla_qubits[stabilizer_index])
            circuit.measure(ancilla_qubits[stabilizer_index], syndrome_bits[round_index * ANCILLA_QUBIT_COUNT + stabilizer_index])
        if round_index < SYNDROME_ROUNDS - 1:
            circuit.reset(ancilla_qubits)

    circuit.measure(data_qubits, data_bits)
    return circuit


def build_bit_flip_noise_model(physical_error_rate):
    """Noise model that turns every id gate into an X error with the given probability"""
    noise_model = NoiseModel()
    bit_flip_channel = pauli_error([("X", physical_error_rate), ("I", 1 - physical_error_rate)])
    noise_model.add_all_qubit_quantum_error(bit_flip_channel, ["id"])
    return noise_model


repetition_circuit = build_repetition_code_circuit()
print(repetition_circuit.draw(fold=140))

In [ ]:
# Run the circuit and unpack every shot into (syndrome bits, true error bits).
#
# qiskit returns counts keyed by a string like '0110 10110' with the LAST-declared register first, and inside each
# register the RIGHTMOST character is bit 0. So the string is '<final data bits> <syndrome bits>', and both halves
# have to be reversed to put bit 0 at index 0.
#
# The circuit is never transpiled. A transpile at any optimization level above 0 would delete the identity gates
# and the noise with them. The mean-flips check below would catch that.

def simulate_shots(physical_error_rate, shots):
    """Returns (syndromes, errors) as integer arrays of shape (shots, SYNDROME_BIT_COUNT) and (shots, DATA_QUBIT_COUNT)"""
    # Every gate here is Clifford and the noise is Pauli, so Aer picks its stabilizer simulator and scales to many qubits.
    simulator = AerSimulator(noise_model=build_bit_flip_noise_model(physical_error_rate),
                             seed_simulator=int(random_generator.integers(0, 2**31)))
    counts = simulator.run(repetition_circuit, shots=shots).result().get_counts()

    syndromes = []
    errors = []
    for outcome_string, count in counts.items():
        final_data_string, syndrome_string = outcome_string.split(" ")
        assert len(syndrome_string) == SYNDROME_BIT_COUNT
        assert len(final_data_string) == DATA_QUBIT_COUNT
        syndrome_bits = [int(character) for character in reversed(syndrome_string)]
        error_bits = [int(character) for character in reversed(final_data_string)]
        for repeat_index in range(count):
            syndromes.append(syndrome_bits)
            errors.append(error_bits)
    syndromes = np.array(syndromes, dtype=np.int64)
    errors = np.array(errors, dtype=np.int64)
    assert syndromes.shape == (shots, SYNDROME_BIT_COUNT)
    assert errors.shape == (shots, DATA_QUBIT_COUNT)
    return syndromes, errors


def syndrome_of_error(error_bits):
    """The syndrome a given error pattern produces under one perfect round"""
    return np.array([error_bits[stabilizer_index] ^ error_bits[stabilizer_index + 1]
                     for stabilizer_index in range(ANCILLA_QUBIT_COUNT)])


start_time = time.time()
training_syndromes, training_errors = simulate_shots(TRAINING_PHYSICAL_ERROR_RATE, TRAINING_SHOTS)
print(f"simulated {TRAINING_SHOTS} training shots at p = {TRAINING_PHYSICAL_ERROR_RATE} in {time.time() - start_time:.1f}s")
print("mean number of flipped data qubits per shot:", round(training_errors.sum(axis=1).mean(), 3),
      " (expected", round(TRAINING_PHYSICAL_ERROR_RATE * DATA_QUBIT_COUNT, 3), ")")

# With perfect single-round measurement, the recorded syndrome must equal the syndrome computed from the recorded error.
if SYNDROME_ROUNDS == 1:
    for shot_index in range(200):
        assert np.array_equal(training_syndromes[shot_index], syndrome_of_error(training_errors[shot_index]))
    print("syndrome consistency check passed on 200 shots")

In [ ]:
# The metrics. All three take a (shots, DATA_QUBIT_COUNT) array of corrections and the matching true errors.

def recovery_failure_rate(corrections, errors):
    """Fraction of shots where correction XOR error is not the all-zeros codeword"""
    residual = corrections ^ errors
    return (residual.sum(axis=1) > 0).mean()


def logical_flip_rate(corrections, errors):
    """Fraction of shots where the residual is exactly all ones, which is a clean logical X error"""
    residual = corrections ^ errors
    return (residual.sum(axis=1) == DATA_QUBIT_COUNT).mean()


def syndrome_consistency_rate(corrections, syndromes):
    """Fraction of corrections whose own syndrome equals the first measured round"""
    correction_syndromes = np.array([syndrome_of_error(correction) for correction in corrections])
    return (correction_syndromes == syndromes[:, :ANCILLA_QUBIT_COUNT]).all(axis=1).mean()


def majority_readout_error_rate(corrections, errors):
    """Classical readout baseline: fraction of shots where the corrected block majority-votes to 1. See the header for why this cannot grade a decoder."""
    residual = corrections ^ errors
    return (residual.sum(axis=1) > DATA_QUBIT_COUNT / 2).mean()


# The claim in the header, checked by brute force: majority readout is identical for minimum-weight correction and
# for no correction, on every one of the 2^d error patterns. The minimum-weight table is built in the next cell, so
# this check runs there.
print("metrics defined")

In [ ]:
# Decoder 1, the pedagogical baseline: minimum-weight lookup table.
# For every possible syndrome, the correction is the lowest-weight error pattern that produces it.
# Built by enumerating all 2^d error patterns, which is fine up to about d = 15 and impossible past d = 25.

minimum_weight_correction = {}
for error_pattern in itertools.product([0, 1], repeat=DATA_QUBIT_COUNT):
    error_pattern = np.array(error_pattern)
    syndrome_key = tuple(int(bit) for bit in syndrome_of_error(error_pattern))
    if syndrome_key not in minimum_weight_correction or error_pattern.sum() < minimum_weight_correction[syndrome_key].sum():
        minimum_weight_correction[syndrome_key] = error_pattern
assert len(minimum_weight_correction) == 2 ** ANCILLA_QUBIT_COUNT


def decode_by_lookup(syndromes):
    """Minimum-weight correction for each syndrome row, using only the first round of syndrome bits"""
    return np.array([minimum_weight_correction[tuple(int(bit) for bit in syndrome[:ANCILLA_QUBIT_COUNT])] for syndrome in syndromes])


# Decoder 2, the scalable exact baseline: the chain decoder. Walk the syndrome from left to right: fix e_0 = 0, then
# e_{i+1} = e_i XOR s_i. That is one of the two error patterns consistent with the syndrome; the other is its
# complement. Return whichever has lower weight. O(d) time, no table. For a one-round perfect syndrome this is exactly
# minimum-weight decoding, and the assert below proves it on every syndrome.

def decode_by_chain(syndromes):
    """Minimum-weight correction by walking the syndrome, no lookup table"""
    corrections = np.zeros((len(syndromes), DATA_QUBIT_COUNT), dtype=np.int64)
    for shot_index, syndrome in enumerate(syndromes):
        candidate = np.zeros(DATA_QUBIT_COUNT, dtype=np.int64)
        for stabilizer_index in range(ANCILLA_QUBIT_COUNT):
            candidate[stabilizer_index + 1] = candidate[stabilizer_index] ^ syndrome[stabilizer_index]
        if candidate.sum() > DATA_QUBIT_COUNT / 2:
            candidate = 1 - candidate
        corrections[shot_index] = candidate
    return corrections


every_syndrome = np.array(list(itertools.product([0, 1], repeat=ANCILLA_QUBIT_COUNT)))
assert np.array_equal(decode_by_lookup(every_syndrome), decode_by_chain(every_syndrome))
print("chain decoder equals the lookup table on all", len(every_syndrome), "syndromes")

# The header's claim about majority readout, checked on every error pattern.
every_error = np.array(list(itertools.product([0, 1], repeat=DATA_QUBIT_COUNT)))
every_error_syndrome = np.array([syndrome_of_error(error) for error in every_error])
no_correction = np.zeros_like(every_error)
assert majority_readout_error_rate(decode_by_lookup(every_error_syndrome), every_error) == majority_readout_error_rate(no_correction, every_error)
print("majority readout is identical with minimum-weight correction and with no correction, on all", len(every_error), "error patterns")

print("lookup table, a few entries (syndrome -> correction):")
for syndrome_key in list(minimum_weight_correction)[:4]:
    print("  ", syndrome_key, "->", minimum_weight_correction[syndrome_key])
print(f"minimum-weight recovery failure on the training shots at p = {TRAINING_PHYSICAL_ERROR_RATE}: "
      f"{recovery_failure_rate(decode_by_lookup(training_syndromes), training_errors):.4f}")
print(f"no-correction recovery failure for comparison (any flip at all):      "
      f"{recovery_failure_rate(np.zeros_like(training_errors), training_errors):.4f}")

In [ ]:
# Decoder 3, the thing this notebook is about: a neural network from syndrome bits to error bits.
# Input: SYNDROME_BIT_COUNT bits. Output: DATA_QUBIT_COUNT independent probabilities that each data qubit flipped.
# Correction = round the probabilities. Trained with binary cross-entropy on the simulated shots.
# Nothing forces its output to be consistent with the syndrome. The metrics will say whether it learned to be.

class NeuralDecoder(torch.nn.Module):
    """Two hidden layers, one sigmoid output per data qubit"""
    def __init__(self):
        super().__init__()
        self.layers = torch.nn.Sequential(
            torch.nn.Linear(SYNDROME_BIT_COUNT, HIDDEN_WIDTH),
            torch.nn.ReLU(),
            torch.nn.Linear(HIDDEN_WIDTH, HIDDEN_WIDTH),
            torch.nn.ReLU(),
            torch.nn.Linear(HIDDEN_WIDTH, DATA_QUBIT_COUNT),
        )

    def forward(self, syndrome_batch):
        return self.layers(syndrome_batch)      # logits, the loss applies the sigmoid


neural_decoder = NeuralDecoder()
optimizer = torch.optim.Adam(neural_decoder.parameters(), lr=LEARNING_RATE)
loss_function = torch.nn.BCEWithLogitsLoss()

training_inputs = torch.tensor(training_syndromes, dtype=torch.float32)
training_targets = torch.tensor(training_errors, dtype=torch.float32)

start_time = time.time()
for epoch_index in range(TRAINING_EPOCHS):
    shuffled_order = torch.randperm(TRAINING_SHOTS)
    epoch_loss = 0.0
    for batch_start in range(0, TRAINING_SHOTS, BATCH_SIZE):
        batch_indices = shuffled_order[batch_start:batch_start + BATCH_SIZE]
        optimizer.zero_grad()
        batch_loss = loss_function(neural_decoder(training_inputs[batch_indices]), training_targets[batch_indices])
        batch_loss.backward()
        optimizer.step()
        epoch_loss += batch_loss.item() * len(batch_indices)
    if (epoch_index + 1) % 5 == 0:
        print(f"epoch {epoch_index + 1}/{TRAINING_EPOCHS}  mean loss {epoch_loss / TRAINING_SHOTS:.4f}  elapsed {time.time() - start_time:.0f}s")


def decode_by_network(syndromes):
    """Rounds the network's per-qubit flip probabilities into a correction"""
    with torch.no_grad():
        logits = neural_decoder(torch.tensor(syndromes, dtype=torch.float32))
    return (logits > 0).numpy().astype(np.int64)


# Audit the network on every possible syndrome: which ones does it decode exactly like minimum weight?
network_on_every_syndrome = decode_by_network(every_syndrome)
agreement = (network_on_every_syndrome == decode_by_lookup(every_syndrome)).all(axis=1)
print(f"network agrees with minimum weight on {agreement.sum()} of {len(every_syndrome)} syndromes")
print(f"network corrections consistent with their syndrome: {syndrome_consistency_rate(network_on_every_syndrome, every_syndrome):.3f} of all syndromes")
print(f"neural recovery failure on the training shots at p = {TRAINING_PHYSICAL_ERROR_RATE}: "
      f"{recovery_failure_rate(decode_by_network(training_syndromes), training_errors):.4f}")

In [ ]:
# Evaluation: fresh shots at every physical error rate, every decoder, every metric.

DECODERS = {
    "no correction": lambda syndromes: np.zeros((len(syndromes), DATA_QUBIT_COUNT), dtype=np.int64),
    "minimum weight": decode_by_chain,
    "neural": decode_by_network,
}
METRICS = {
    "recovery failure": recovery_failure_rate,
    "logical flip": logical_flip_rate,
    "majority readout": majority_readout_error_rate,
}

evaluation_results = {"physical_error_rate": np.array(EVALUATION_PHYSICAL_ERROR_RATES)}
for decoder_name in DECODERS:
    for metric_name in METRICS:
        evaluation_results[(decoder_name, metric_name)] = []
    evaluation_results[(decoder_name, "syndrome consistency")] = []
    evaluation_results[(decoder_name, "failure count")] = []

start_time = time.time()
for physical_error_rate in EVALUATION_PHYSICAL_ERROR_RATES:
    syndromes, errors = simulate_shots(physical_error_rate, EVALUATION_SHOTS)
    for decoder_name, decoder in DECODERS.items():
        corrections = decoder(syndromes)
        for metric_name, metric in METRICS.items():
            evaluation_results[(decoder_name, metric_name)].append(metric(corrections, errors))
        evaluation_results[(decoder_name, "syndrome consistency")].append(syndrome_consistency_rate(corrections, syndromes))
        evaluation_results[(decoder_name, "failure count")].append(int(((corrections ^ errors).sum(axis=1) > 0).sum()))
    print(f"p = {physical_error_rate:.2f}   recovery failure: "
          + "   ".join(f"{decoder_name} {evaluation_results[(decoder_name, 'recovery failure')][-1]:.4f}" for decoder_name in DECODERS)
          + f"   elapsed {time.time() - start_time:.0f}s")
for key in evaluation_results:
    evaluation_results[key] = np.array(evaluation_results[key])

# Exact reference for minimum-weight decoding under independent bit flips: it fails exactly when more than half the
# data qubits flip, so the failure probability is the binomial upper tail.
def exact_minimum_weight_failure(physical_error_rate):
    """Probability that at least (d+1)/2 of d data qubits flip"""
    return binom.sf((DATA_QUBIT_COUNT - 1) / 2, DATA_QUBIT_COUNT, physical_error_rate)


training_rate_position = EVALUATION_PHYSICAL_ERROR_RATES.index(TRAINING_PHYSICAL_ERROR_RATE)
print()
print(f"RESULTS AT THE TRAINING RATE p = {TRAINING_PHYSICAL_ERROR_RATE}, distance {CODE_DISTANCE}, {EVALUATION_SHOTS} shots")
print(f"{'decoder':16s}{'recovery failure':>18s}{'logical flip':>14s}{'syndrome consistent':>21s}{'majority readout':>18s}")
for decoder_name in DECODERS:
    print(f"{decoder_name:16s}{evaluation_results[(decoder_name, 'recovery failure')][training_rate_position]:18.4f}"
          f"{evaluation_results[(decoder_name, 'logical flip')][training_rate_position]:14.4f}"
          f"{evaluation_results[(decoder_name, 'syndrome consistency')][training_rate_position]:21.3f}"
          f"{evaluation_results[(decoder_name, 'majority readout')][training_rate_position]:18.4f}")
print(f"{'exact min weight':16s}{exact_minimum_weight_failure(TRAINING_PHYSICAL_ERROR_RATE):18.4f}   (binomial tail, no sampling)")
print(f"{'unencoded qubit':16s}{TRAINING_PHYSICAL_ERROR_RATE:18.4f}")
print()
print("Read the majority-readout column across the three decoders. It is the same number for all of them. That is why it is a readout baseline and not a decoder score.")

In [ ]:
# Save everything a follow-up notebook needs.

config = {
    "RANDOM_SEED": RANDOM_SEED,
    "CODE_DISTANCE": CODE_DISTANCE,
    "SYNDROME_ROUNDS": SYNDROME_ROUNDS,
    "TRAINING_PHYSICAL_ERROR_RATE": TRAINING_PHYSICAL_ERROR_RATE,
    "EVALUATION_PHYSICAL_ERROR_RATES": EVALUATION_PHYSICAL_ERROR_RATES,
    "TRAINING_SHOTS": TRAINING_SHOTS,
    "EVALUATION_SHOTS": EVALUATION_SHOTS,
    "HIDDEN_WIDTH": HIDDEN_WIDTH,
    "TRAINING_EPOCHS": TRAINING_EPOCHS,
    "BATCH_SIZE": BATCH_SIZE,
    "LEARNING_RATE": LEARNING_RATE,
    "package_versions": package_versions,
}
dataset = {
    "config": config,
    "training_syndromes": training_syndromes,
    "training_errors": training_errors,
    "minimum_weight_correction": minimum_weight_correction,
    "neural_decoder_state_dict": neural_decoder.state_dict(),
    "network_on_every_syndrome": network_on_every_syndrome,
    "evaluation_results": evaluation_results,
}
with open(OUTPUT_PATH, "wb") as output_file:
    pickle.dump(dataset, output_file)
print("saved", OUTPUT_PATH)

In [ ]:
# Poster figure: recovery failure versus physical error rate, log-log, with the unencoded line and the exact curve.
# A rate estimated from zero observed failures cannot be drawn on a log axis. Those points are drawn as open
# triangles at the 95% upper bound 3 / EVALUATION_SHOTS (the rule of three), which is what the data support.

rates = evaluation_results["physical_error_rate"]
fine_rates = np.logspace(np.log10(rates.min()), np.log10(rates.max()), 100)
upper_bound_for_zero_failures = 3 / EVALUATION_SHOTS

figure, axis = plt.subplots(figsize=(7.5, 5.5))
axis.plot(rates, rates, "k--", linewidth=0.8, label="unencoded (failure = physical)")
axis.plot(fine_rates, exact_minimum_weight_failure(fine_rates), color="gray", linewidth=1.0, label="exact minimum weight (binomial tail)")
for decoder_name, marker in zip(["minimum weight", "neural", "no correction"], ["o", "s", "x"]):
    failure_rates = evaluation_results[(decoder_name, "recovery failure")]
    failure_counts = evaluation_results[(decoder_name, "failure count")]
    observed = failure_counts > 0
    axis.plot(rates[observed], failure_rates[observed], marker=marker, linestyle="-", label=decoder_name)
    if (~observed).any():
        axis.plot(rates[~observed], np.full((~observed).sum(), upper_bound_for_zero_failures), marker="v",
                  linestyle="none", markerfacecolor="none", color=axis.lines[-1].get_color(),
                  label=f"{decoder_name}: no failures seen, 95% upper bound")
axis.axvline(TRAINING_PHYSICAL_ERROR_RATE, color="gray", linewidth=0.8, linestyle=":", label="neural training rate")
axis.set_xscale("log")
axis.set_yscale("log")
axis.set_xlabel("physical bit-flip rate")
axis.set_ylabel("recovery failure rate")
axis.set_title(f"Distance-{CODE_DISTANCE} repetition code, {EVALUATION_SHOTS} shots per point: logical-error suppression")
axis.legend(fontsize=8)
axis.grid(True, which="both", linewidth=0.3)
figure.tight_layout()
figure.savefig(FIGURE_PATH, dpi=200)
plt.show()
print("saved", FIGURE_PATH)
print(f"expected failures at p = {rates.min()} with minimum weight: {EVALUATION_SHOTS * exact_minimum_weight_failure(rates.min()):.2f}; zero observed is likely there")

## Shred-outs

The base run shows a neural decoder reaching the optimal decoder on the easiest possible problem. Directions 2, 3, and 5 change the problem so that minimum-weight decoding from one clean syndrome is no longer optimal, which is where a learned decoder can earn its keep. Direction 1 does not break any assumption; it tests scaling. Pick one, or invent your own. Each is written as the question your poster answers and the number that answers it. Use the recovery failure rate and the syndrome consistency rate, not majority readout, and show a confidence bound wherever you observe zero failures. If your extension sweeps a setting, choose it on validation shots and report the test number once.

1. **Distance sweep.** Run distances 3, 5, 7, 9, 11. The lookup table grows as 2^(d-1), the chain decoder is O(d), and the neural decoder's input grows linearly. Does the neural decoder keep pace at every distance, and does the suppression curve steepen the way the binomial tail says it should? Metric: recovery failure at the training rate versus distance, all decoders, plus the full curve at each distance. Minimum weight stays optimal throughout this sweep; the question is whether learning keeps up, not whether it wins.

2. **Noisy measurement, many rounds.** Add a readout error to the ancilla measurements (`qiskit_aer.noise.ReadoutError` on the ancilla qubits, added per qubit) and set `SYNDROME_ROUNDS` to 3 or 5. A single flipped syndrome bit now has two explanations, a data error or a measurement error, and a decoder working from one round is wrong some of the time. Two baselines are needed: the chain decoder on the first round only, and the chain decoder on the majority vote over rounds of each syndrome bit. Feed the neural decoder the whole syndrome history. Metric: recovery failure versus measurement error rate, all three.

3. **Correlated noise.** Replace the independent bit-flip channel with one that flips neighboring pairs together some fraction of the time (`pauli_error` on two qubits with an `XX` term, attached to a two-qubit identity). Minimum-weight decoding assumes independence and will choose the wrong correction for pair flips. Does the neural decoder learn the correlation? Metric: recovery failure versus the fraction of correlated errors, both decoders.

4. **Train at one rate, decode at others.** The base run trains at p = 0.10 and tests from 0.01 to 0.40. Where does the neural decoder fall behind minimum weight, and does training on a mixture of rates fix it? Metric: the gap between neural and minimum-weight recovery failure across the sweep, for single-rate and mixed-rate training.

5. **Circuit-level noise.** Put depolarizing noise on the CX gates instead of on the data qubits (`depolarizing_error` on `cx`). Be precise about what that does here. In this extraction circuit the data qubits are CX controls and the ancillas are targets. An X error on a data qubit between its two stabilizer checks is seen by one check and not the other, so the syndrome disagrees with the final readout the way a measurement error would. An X error on an ancilla flips one syndrome bit and never reaches the data. Z errors on either side are invisible to the Z-basis readout and unprotected by this code anyway. Metric: recovery failure versus CX error rate, all decoders, and the syndrome consistency rate, which will drop for the chain decoder.

6. **Data efficiency.** Sweep `TRAINING_SHOTS` from 100 to 100000. How many shots does the neural decoder need to match minimum weight, and how does that number grow with distance? Metric: recovery failure versus training shots, one curve per distance.

7. **A real surface code patch.** Build the distance-3 rotated surface code (9 data qubits, 8 ancillas) with both X-type and Z-type stabilizers. The free ground-truth trick in this notebook only works for one error type at a time: inject X errors and read the data out in the Z basis to label them, or inject Z errors and read out in the X basis, in separate runs. The stabilizer simulator handles the circuit easily. Two error patterns that differ by a stabilizer have the same syndrome and the same logical effect, so a decoder must be scored on the logical class of its residual, not on exact recovery, and a minimum-weight baseline needs a matching algorithm. Metric: logical failure rate versus physical rate for the surface code, neural against minimum-weight matching.

Whatever you choose, load `repetition_decoder_dataset.pkl` for the base-run shots, the trained decoder weights, the network's output on every syndrome, and the evaluation curves. The `config` dict inside tells you exactly which run produced it.

In [ ]:
# Your extension starts here.
